# BO Forge minimisation campaign

Short demo for a minimisation problem. The workflow starts from two manual observations, fills the Sobol initial design, runs one qLogEI batch BO round, and confirms that best-so-far means the lowest user-facing objective value.

In [ ]:
from pathlib import Path
import os
import shutil
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

mpl_cache = PROJECT_ROOT / ".matplotlib-cache"
mpl_cache.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from bo_forge import CampaignConfig, append_suggestions, load_campaign_log, mark_observed, suggest_next
from bo_forge.diagnostics import plot_progress
from bo_forge.validation import get_observed_data, validate_campaign_data

## Load config and seed log

In [ ]:
config_path = PROJECT_ROOT / "configs" / "simple_2d_minimize.yaml"
seed_log_path = PROJECT_ROOT / "examples" / "simple_2d_minimize_campaign_log.csv"
working_log_path = PROJECT_ROOT / "examples" / "simple_2d_minimize_working_log.csv"

shutil.copyfile(seed_log_path, working_log_path)

config = CampaignConfig.from_yaml(config_path)
df = load_campaign_log(working_log_path, config)
validate_campaign_data(config, df)

print(config)
display(df)

## Simulate one experiment at a time

In [ ]:
def simulated_defect_rate(row) -> float:
    loading = float(row["catalyst_loading"])
    temperature = float(row["cure_temperature"])
    defect_rate = 0.08 + 2.2 * (loading - 0.38) ** 2
    defect_rate += ((temperature - 128.0) / 38.0) ** 2
    defect_rate += 0.015 * np.sin(9.0 * loading)
    defect_rate += 0.01 * np.cos(temperature / 18.0)
    return float(defect_rate)


def current_log():
    df = load_campaign_log(working_log_path, config)
    validate_campaign_data(config, df)
    return df


def suggest_and_observe():
    df = current_log()
    suggestion = suggest_next(config, df, batch_size=1)
    append_suggestions(working_log_path, suggestion)

    row = suggestion.iloc[0]
    result = simulated_defect_rate(row)
    mark_observed(working_log_path, str(row["row_id"]), result)
    return suggestion, result, current_log()


def suggest_batch_and_observe(batch_size: int):
    df = current_log()
    suggestions = suggest_next(config, df, batch_size=batch_size)
    append_suggestions(working_log_path, suggestions)

    records = []
    for _, row in suggestions.iterrows():
        result = simulated_defect_rate(row)
        mark_observed(working_log_path, str(row["row_id"]), result)
        records.append(compact_record(row, result))
    return suggestions, records, current_log()


def compact_record(row, result):
    return {
        "source": row["source"],
        "iteration": int(row["iteration"]),
        "catalyst_loading": float(row["catalyst_loading"]),
        "cure_temperature": float(row["cure_temperature"]),
        "defect_rate": result,
    }

## Fill the Sobol initial design

In [ ]:
sobol_records = []
while len(get_observed_data(config, current_log())) < config.bo.initial_design_size:
    suggestion, result, df = suggest_and_observe()
    sobol_records.append(compact_record(suggestion.iloc[0], result))

sobol_df = pd.DataFrame(sobol_records)
assert set(sobol_df["source"]) == {"sobol"}
sobol_df

## Run one qLogEI batch BO round

In [ ]:
batch_suggestions, bo_records, df = suggest_batch_and_observe(config.bo.batch_size)
bo_df = pd.DataFrame(bo_records)

assert config.bo.batch_size > 1
assert set(bo_df["source"]) == {"qlog_ei"}

display(batch_suggestions)
bo_df

## Plot progress and confirm the minimisation best

In [ ]:
df = current_log()
observed = get_observed_data(config, df)
values = pd.to_numeric(observed[config.objective.name])
best_index = values.idxmin()
best_row = observed.loc[[best_index], ["row_id", *config.variable_names, config.objective.name]]

print(f"Objective direction: {config.objective.direction}")
print(f"Best means lowest {config.objective.name}: {values.min():.6f}")
display(best_row)

assert config.objective.direction == "minimize"
assert values.min() == values.cummin().iloc[-1]

plot_progress(config, df);